# Experiment

> This notebook reproduces the results presented in *nanoTabPFN: A Lightweight and Educational Reimplementation of TabPFN*.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import functools
from collections import defaultdict
from collections.abc import Callable, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from tabpfn import TabPFNClassifier
from tabpfn.preprocessing import PreprocessorConfig

from nanotabpfn import (
    NanoTabPFNModel,
    PriorDumpDataLoader,
    get_default_device,
    set_randomness_seed,
    train,
)
from nanotabpfn.experiment import (
    EvalResults,
    evaluate_model,
    get_openml_datasets,
    plot_nano_runs,
    plot_nano_runs_v2,
    plot_run_grid,
    plot_run_grid_v2,
)


def apply_aggregation(
    eval_results: list[EvalResults],
    func: Callable[[Sequence[float]], float | np.floating[Any]] = np.mean,
) -> dict[str, Any]:

    results_by_metric = defaultdict(list)

    for result in eval_results:
        for metric, value in result.items():
            results_by_metric[metric].append(value)

    aggregated_results = {metric: func(values) for metric, values in results_by_metric.items()}

    return aggregated_results

## Training

In [ ]:
NUM_SEEDS = 2
# NUM_SEEDS = 20 # If you want to reproduce the paper results, use 20 seeds
DATASETS = get_openml_datasets(max_features=10, new_instances=200, target_classes_filter=2)

In [ ]:
RESULTS_DIR = Path("./experiment-results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINES_PATH = RESULTS_DIR / "baselines.pkl"
NANO_RUNS_PATH = RESULTS_DIR / "nano-runs.pkl"

LOAD_BASELINES = True
LOAD_HISTORIES = True
# LOAD_BASELINES = False
# LOAD_HISTORIES = False

### "baselines" - tabpfn and others

In [ ]:
baseline_models = {
    "TabPFN v2": [TabPFNClassifier(random_state=i) for i in range(NUM_SEEDS)],
    "TabPFN v2 (no preprocessing)": [
        TabPFNClassifier(
            inference_config={
                "PREPROCESS_TRANSFORMS": [PreprocessorConfig(name="none")],
                "FINGERPRINT_FEATURE": False,
            },
            n_estimators=1,
            random_state=i,
        )
        for i in range(NUM_SEEDS)
    ],
    "Random Forest": [RandomForestClassifier(random_state=i) for i in range(NUM_SEEDS)],
    "K-Nearest Neighbors": [KNeighborsClassifier()],
    "Decision Tree": [DecisionTreeClassifier(random_state=i) for i in range(NUM_SEEDS)],
}

In [ ]:
if LOAD_BASELINES:
    baseline_results: dict[str, list[EvalResults]] = pd.read_pickle(BASELINES_PATH)  # ty: ignore[invalid-assignment]
else:
    baseline_results = {
        name: [evaluate_model(model, datasets=DATASETS) for model in models]
        for name, models in baseline_models.items()
    }

    pd.to_pickle(
        baseline_results,
        BASELINES_PATH,
    )

baselines_mean = pd.DataFrame(
    {name: apply_aggregation(models, np.mean) for name, models in baseline_results.items()}
).T

baselines_mean_v2 = pl.from_pandas(baselines_mean).with_columns(
    baseline=pl.Series(baselines_mean.index.tolist())
)

baselines_std = pd.DataFrame(
    {name: apply_aggregation(models, np.std) for name, models in baseline_results.items()}
).T

### nanotabpfn

In [ ]:
if LOAD_HISTORIES:
    nano_runs: list[pd.DataFrame] = pd.read_pickle(NANO_RUNS_PATH)  # ty: ignore[invalid-assignment]
    nano_runs_v2 = [pl.from_pandas(df) for df in nano_runs]
else:
    device = get_default_device()
    nano_runs = []

    for seed in range(NUM_SEEDS):
        set_randomness_seed(seed)
        prior = PriorDumpDataLoader("300k_150x5_2.h5", num_steps=2500, batch_size=32, device=device)
        model = NanoTabPFNModel(
            embedding_size=96,
            num_attention_heads=4,
            mlp_hidden_size=192,
            num_layers=3,
            num_outputs=2,
        )
        eval_model_partial = functools.partial(evaluate_model, datasets=DATASETS)
        model, history = train(
            model, prior, lr=4e-3, steps_per_eval=25, eval_func=eval_model_partial
        )
        history = [
            {**record, "training_time": training_time} for (training_time, record) in history
        ]
        history = pd.DataFrame(history)
        nano_runs.append(history)

    pd.to_pickle(nano_runs, NANO_RUNS_PATH)

## Eval

In [ ]:
metric = "ROC AUC"
sns.set_style("ticks")
sns.set_context("notebook")
fig, ax = plt.subplots(figsize=(6, 4.5), layout="constrained")
plot_nano_runs_v2(ax, nano_runs_v2, metric, baselines_mean_v2)

In [ ]:
sns.set_context("notebook")
fig, ax = plot_run_grid_v2(nano_runs_v2, baselines_mean_v2)
plt.show()
# fig.savefig("nanotabpfn_summary.png", dpi=300)